In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

data = pd.read_csv('Thursday.csv')
df = data.copy()
df['Label'] = df['Label'].map(lambda x: 0 if x == "BENIGN" else 1)


print("Unikalne wartości w kolumnie 'Label':")
print(df['Label'].unique())
print(df['Label'].value_counts())

df.columns = df.columns.str.replace(' ', '_')
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

X = df.drop('Label', axis=1).values
y = df['Label'].values

scaler = StandardScaler()
X_normalized = scaler.fit_transform(X) 

X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42, stratify=y)

input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))

encoded = Dense(128, activation='relu')(input_layer)
encoded = Dense(64, activation='relu')(encoded)
encoded = Dense(32, activation='relu')(encoded) 

decoded = Dense(64, activation='relu')(encoded)
decoded = Dense(128, activation='relu')(decoded)

output_layer = Dense(1, activation='sigmoid')(decoded)

model = Model(inputs=input_layer, outputs=output_layer)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'Precision', 'Recall'])

model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

y_pred_probs = model.predict(X_test)
y_pred_classes = (y_pred_probs >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred_classes)
print("Confusion Matrix:")
print(cm)
print("Classification Report:")
print(classification_report(y_test, y_pred_classes, target_names=['BENIGN', 'ATTACK']))

Unikalne wartości w kolumnie 'Label':
[0 1]
Label
0    456752
1      2216
Name: count, dtype: int64
Epoch 1/5
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 7s 587us/step - Precision: 0.4433 - Recall: 0.4509 - accuracy: 0.9941 - loss: 0.0180 - val_Precision: 0.5941 - val_Recall: 0.9481 - val_accuracy: 0.9966 - val_loss: 0.0083
Epoch 2/5
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 7s 613us/step - Precision: 0.6211 - Recall: 0.8731 - accuracy: 0.9967 - loss: 0.0080 - val_Precision: 0.7937 - val_Recall: 0.1129 - val_accuracy: 0.9956 - val_loss: 0.0086
Epoch 3/5
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 7s 609us/step - Precision: 0.6422 - Recall: 0.8801 - accuracy: 0.9970 - loss: 0.0066 - val_Precision: 0.5941 - val_Recall: 0.9481 - val_accuracy: 0.9966 - val_loss: 0.0065
Epoch 4/5
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 6s 556us/step - Precision: 0.6228 - Recall: 0.8863 - accuracy: 0.9968 - loss: 0.0092 - val_Precision: 0.5877 - val_Recall: 0.9526 - val_accuracy: 0.9965 - val_loss: 0.0074
Epoch 5/5
11466/11466 ━━━━━━━━━━━━━━━━━━

```
[[90994   289]
 [   21   422]]
```
- **True Negatives (TN)**: 90,994 — poprawnie sklasyfikowane `BENIGN`.
- **False Positives (FP)**: 289 — `BENIGN` błędnie sklasyfikowane jako `ATTACK`.
- **False Negatives (FN)**: 21 — `ATTACK` błędnie sklasyfikowane jako `BENIGN`.
- **True Positives (TP)**: 422 — poprawnie wykryte `ATTACK`.

#### **Metryki dla klasy `ATTACK`**
- **Precision**: `0.59`  
  Tylko 59% wyników przewidzianych jako `ATTACK` jest rzeczywiście atakami. Model generuje stosunkowo dużo fałszywych alarmów (`FP` = 289).
- **Recall**: `0.95`  
  Model wykrywa 95% wszystkich ataków (`ATTACK`), co oznacza, że bardzo rzadko pomija rzeczywiste ataki (`FN` = 21).
- **F1-score**: `0.73`  
  Jest to harmoniczna średnia precision i recall. Wynik `0.73` wskazuje, że model ma problemy z precyzją, ale dobrze radzi sobie z wykrywaniem ataków.

#### **Ogólna dokładność (accuracy)**: `1.00`
- Wysoka dokładność wynika z dominacji klasy `BENIGN` w zbiorze danych. Model dobrze radzi sobie z klasyfikacją tej klasy, co zawyża accuracy.

### Analiza wyników

1. **Precision dla `ATTACK` jest niska (`0.59`)**:
   - Model generuje stosunkowo dużo fałszywych alarmów (`FP` = 289). To może być problematyczne w praktycznych zastosowaniach, gdzie fałszywe alarmy są kosztowne.

2. **Recall dla `ATTACK` jest wysoki (`0.95`)**:
   - Model wykrywa prawie wszystkie ataki, co jest bardzo pozytywne. Tylko 21 ataków zostało pominiętych (`FN` = 21).

3. **F1-score dla `ATTACK` (`0.73`)**:
   - Wynik wskazuje na pewną nierównowagę między precision i recall. Model dobrze wykrywa ataki, ale ma problemy z precyzją.

4. **Ogólna dokładność (`1.00`)**:
   - Wysoka dokładność wynika z dominacji klasy `BENIGN`. Nie jest to najlepsza metryka w przypadku niezbalansowanych danych.

